# Gold: modelo estrela

Este notebook transforma as tabelas Silver em dimensoes analiticas e na tabela fato `workspace.olist_gold.fato_vendas`, com granularidade de um item por pedido.

A fato combina pedidos, itens, avaliacoes e o primeiro pagamento de cada pedido. A coluna `delivery_status` classifica o cumprimento do prazo estimado.

**Mudancas para as novas hipoteses (ver `.copilot/catalogo_dados.md`):**
- `dim_produtos` ganha peso e dimensoes fisicas (Pergunta 1 — frete vs. porte fisico).
- Nova dimensao `dim_vendedores` (Pergunta 2 — concentracao de problemas por vendedor).
- `fato_vendas` ganha `seller_id`, `review_id`, `payment_installments`, `payment_value`, `delivery_date` e `order_estimated_delivery_date`; `purchase_date` vira `order_date` e `delivery_delay_days` vira `dias_vs_estimativa`.
- `dim_avaliacoes` passa a expor `rating`/`rating_label` (Excelente/Bom/Aceitavel/Ruim) em vez do rotulo generico `nota_N`.
- `dim_tempo` foi removida: o modelo-alvo tem 1 fato + 4 dimensoes, e `order_date` ja fica disponivel na propria fato para qualquer corte temporal.

**Ordem de execucao:** execute o Bronze e o Silver antes deste notebook.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Gold - modelo estrela

# COMMAND ----------

from pyspark.sql import Window, functions as F

CATALOG = "workspace"
SILVER = f"{CATALOG}.olist_silver"
GOLD = f"{CATALOG}.olist_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")


def save_gold(dataframe, table_name):
    dataframe.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(f"{GOLD}.{table_name}")
    print(f"✅ {GOLD}.{table_name}: {dataframe.count()} registros")


# Dimensoes
customers = spark.table(f"{SILVER}.customers")
dim_clientes = customers.select(
    "customer_id",
    F.col("customer_state").alias("state"),
    F.col("customer_city").alias("city"),
    F.col("customer_zip_code_prefix").alias("zip_code"),
).dropDuplicates(["customer_id"])
save_gold(dim_clientes, "dim_clientes")

products = spark.table(f"{SILVER}.products")
dim_produtos = products.select(
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
).dropDuplicates(["product_id"])
save_gold(dim_produtos, "dim_produtos")

sellers = spark.table(f"{SILVER}.sellers")
dim_vendedores = sellers.select(
    "seller_id",
    F.col("seller_state").alias("state"),
    F.col("seller_city").alias("city"),
).dropDuplicates(["seller_id"])
save_gold(dim_vendedores, "dim_vendedores")

# Dominio fixo de notas validas, com rotulo de satisfacao (independente das avaliacoes observadas).
dim_avaliacoes = spark.range(1, 6).select(
    F.col("id").cast("int").alias("rating")
).withColumn(
    "rating_label",
    F.when(F.col("rating") == 5, "Excelente")
    .when(F.col("rating") == 4, "Bom")
    .when(F.col("rating") == 3, "Aceitável")
    .otherwise("Ruim"),
)
save_gold(dim_avaliacoes, "dim_avaliacoes")

# Mantem apenas o primeiro pagamento por pedido para evitar duplicar itens.
payments = spark.table(f"{SILVER}.payments")
payment_window = Window.partitionBy("order_id").orderBy(
    F.col("payment_sequential").asc_nulls_last(), F.col("payment_type")
)
primary_payment = payments.withColumn(
    "row_number", F.row_number().over(payment_window)
).filter(F.col("row_number") == 1).select(
    "order_id", "payment_type", "payment_installments", "payment_value"
)

orders = spark.table(f"{SILVER}.orders")
items = spark.table(f"{SILVER}.order_items")
reviews = spark.table(f"{SILVER}.reviews").select("order_id", "review_id", "review_score")

fato_vendas = (
    orders.alias("o")
    .join(items.alias("i"), F.col("i.order_id") == F.col("o.order_id"))
    .join(reviews.alias("r"), F.col("r.order_id") == F.col("o.order_id"), "left")
    .join(primary_payment.alias("p"), F.col("p.order_id") == F.col("o.order_id"), "left")
    .filter(
        F.col("o.order_purchase_timestamp").isNotNull()
        & (
            F.col("o.order_delivered_customer_date").isNull()
            | (
                F.col("o.order_delivered_customer_date")
                >= F.col("o.order_purchase_timestamp")
            )
        )
    )
    .select(
        F.col("o.order_id"),
        F.col("i.order_item_id"),
        F.col("o.customer_id"),
        F.col("i.product_id"),
        F.col("i.seller_id"),
        F.col("r.review_id"),
        F.col("r.review_score"),
        F.col("p.payment_type"),
        F.col("p.payment_installments"),
        F.col("p.payment_value"),
        F.col("i.price"),
        F.col("i.freight_value"),
        F.to_date("o.order_purchase_timestamp").alias("order_date"),
        F.col("o.order_delivered_customer_date").alias("delivery_date"),
        F.col("o.order_estimated_delivery_date"),
        F.datediff(
            "o.order_delivered_customer_date", "o.order_purchase_timestamp"
        ).alias("delivery_days"),
        F.datediff(
            "o.order_delivered_customer_date", "o.order_estimated_delivery_date"
        ).alias("dias_vs_estimativa"),
        F.when(
            F.col("o.order_delivered_customer_date").isNull(), "nao_entregue"
        )
        .when(
            F.datediff(
                "o.order_delivered_customer_date", "o.order_estimated_delivery_date"
            )
            <= 0,
            "adiantado_ou_no_prazo",
        )
        .when(
            F.datediff(
                "o.order_delivered_customer_date", "o.order_estimated_delivery_date"
            )
            <= 7,
            "atraso_1_7_dias",
        )
        .otherwise("atraso_mais_de_7_dias")
        .alias("delivery_status"),
    )
)
save_gold(fato_vendas, "fato_vendas")

print("\n✅ GOLD LAYER COMPLETO!")
